In [3]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rag'
REPO = 'https://github.com/Kluey/RAG-DPR-Reproduction'
!if [ -d /content/RAG-DPR-Reproduction/.git ]; then git -C /content/RAG-DPR-Reproduction pull -q; else git clone -q $REPO /content/RAG-DPR-Reproduction; fi
!git -C /content/RAG-DPR-Reproduction log --oneline -1
%cd /content/RAG-DPR-Reproduction
import os, shutil
for d in ('checkpoints', 'results'):
    target = f'{DRIVE}/{d}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(d):
        os.remove(d)
    elif os.path.isdir(d):
        shutil.copytree(d, target, dirs_exist_ok=True)
        shutil.rmtree(d)
    os.symlink(target, d)
    print(d, '->', os.readlink(d), os.listdir(d))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bd63c35 (HEAD -> main, origin/main, origin/HEAD) reafctor: file structure reorg
/content/RAG-DPR-Reproduction
total 8
drwxr-xr-x 2 root root 4096 Oct 15 10:40 .
drwxr-xr-x 1 root root 4096 Oct 15 10:40 ..


In [4]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.9 MB/s eta 0:00:00:00:0100:01


Downloads the data from Hugging Face and rebuilds it in each new session.

In [5]:
!python -m ragdpr.prepare_data --n_train 10000 --n_dev 1000


default/train/0000.parquet: downloading bytes:  95% 284M/298M [00:02<00:00, 97.2MB/s, 24.1MB/s  ] 
default/train/0000.parquet: downloading bytes: 100% 298M/298M [00:03<00:00, 83.0MB/s, 24.1MB/s  ]
default/train/0000.parquet: reconstructing file: 100% 298M/298M [00:03<00:00, 82.9MB/s, 27.2MB/s  ]

default/train/0001.parquet: downloading bytes:  79% 236M/298M [00:02<00:00, 99.2MB/s, 19.9MB/s  ] 
default/train/0001.parquet: downloading bytes:  80% 298M/373M [00:05<00:02, 33.4MB/s, 20.7MB/s  ]
default/train/0001.parquet: downloading bytes: 100% 298M/298M [00:05<00:00, 52.8MB/s, 21.1MB/s  ] ]
default/train/0001.parquet: reconstructing file: 100% 298M/298M [00:05<00:00, 52.8MB/s, 24.2MB/s  ]

default/train/0002.parquet: downloading bytes:  74% 220M/298M [00:01<00:00, 326MB/s, 15.7MB/s  ]  
default/train/0002.parquet: downloading bytes:  92% 273M/298M [00:01<00:00, 187MB/s, 24.5MB/s  ]
default/train/0002.parquet: downloading bytes: 100% 298M/298M [00:02<00:00, 139MB/s, 26.5MB/s  ] ]
default/

In [ ]:
# Default 10 epochs
# !python -m ragdpr.dpr.train --fp16

config.json: 100% 570/570 [00:00<00:00, 3.13MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 288kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 8.45MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 28.2MB/s]

model.safetensors: downloading bytes:  56% 248M/440M [00:00<00:00, 448MB/s, 17.8MB/s  ] 
model.safetensors: downloading bytes:  71% 312M/440M [00:00<00:00, 503MB/s, 24.0MB/s  ]
model.safetensors: downloading bytes:  91% 400M/440M [00:01<00:00, 475MB/s, 35.3MB/s  ] ]
model.safetensors: downloading bytes: 100% 415M/415M [00:01<00:00, 283MB/s, 39.1MB/s  ] ]
model.safetensors: reconstructing file: 100% 440M/440M [00:01<00:00, 300MB/s, 42.0MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5401.11it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.

In [7]:
!python -m ragdpr.dpr.train --epochs 3 --fp16

Loading weights: 100% 199/199 [00:00<00:00, 5083.76it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 199/199 [00:00<00:00, 5701.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-----------------------

In [8]:
!python -m ragdpr.dpr.index --build --eval

Loading weights: 100% 199/199 [00:00<00:00, 4706.03it/s]
encoding 842446 passages on cuda
  encoded 128/842446
  encoded 12928/842446
  encoded 25728/842446
  encoded 38528/842446
  encoded 51328/842446
  encoded 64128/842446
  encoded 76928/842446
  encoded 89728/842446
  encoded 102528/842446
  encoded 115328/842446
  encoded 128128/842446
  encoded 140928/842446
  encoded 153728/842446
  encoded 166528/842446
  encoded 179328/842446
  encoded 192128/842446
  encoded 204928/842446
  encoded 217728/842446
  encoded 230528/842446
  encoded 243328/842446
  encoded 256128/842446
  encoded 268928/842446
  encoded 281728/842446
  encoded 294528/842446
  encoded 307328/842446
  encoded 320128/842446
  encoded 332928/842446
  encoded 345728/842446
  encoded 358528/842446
  encoded 371328/842446
  encoded 384128/842446
  encoded 396928/842446
  encoded 409728/842446
  encoded 422528/842446
  encoded 435328/842446
  encoded 448128/842446
  encoded 460928/842446
  encoded 473728/842446
  encode

Train and evaluate RAG Token, RAG Sequence and BART closed book

In [9]:
!python -m ragdpr.rag.train --variant seq --fp16
!python -m ragdpr.rag.train --variant token --fp16
!python -m ragdpr.closed_book --fp16

Loading weights: 100% 199/199 [00:00<00:00, 5148.61it/s]
config.json: 100% 1.72k/1.72k [00:00<00:00, 5.88MB/s]

model.safetensors: downloading bytes:   1% 5.08M/558M [00:00<00:56, 9.79MB/s]
model.safetensors: downloading bytes:   6% 34.2M/558M [00:00<00:07, 70.5MB/s,  501kB/s  ]
model.safetensors: downloading bytes:  12% 65.1M/558M [00:00<00:03, 126MB/s, 3.37MB/s  ]  ]
model.safetensors: downloading bytes:  28% 155M/558M [00:00<00:01, 271MB/s, 10.6MB/s  ]  ] 
model.safetensors: reconstructing file:  41% 230M/558M [00:01<00:00, 365MB/s, 7.35MB/s  ] 
model.safetensors: downloading bytes:  44% 245M/558M [00:01<00:00, 347MB/s, 17.6MB/s  ] ]
model.safetensors: downloading bytes:  67% 371M/558M [00:01<00:00, 325MB/s, 31.9MB/s  ] ]
model.safetensors: downloading bytes: 100% 372M/372M [00:01<00:00, 197MB/s, 34.3MB/s  ] ]
model.safetensors: reconstructing file: 100% 558M/558M [00:01<00:00, 296MB/s, 51.9MB/s  ]
Loading weights: 100% 259/259 [00:00<00:00, 5466.89it/s]
vocab.json: 100% 899k/899k [

In [10]:
!python -m ragdpr.rag.evaluate --no_retrieval
!python -m ragdpr.rag.evaluate --variant seq
!python -m ragdpr.rag.evaluate --variant token

Loading weights: 100% 260/260 [00:00<00:00, 3908.39it/s]
closed-book BART: EM = 2.5% (25/1000)
saved exact_match.closed_book to results/results.json
Loading weights: 100% 199/199 [00:00<00:00, 4975.78it/s]
Loading weights: 100% 260/260 [00:00<00:00, 5548.25it/s]
  8/1000
  16/1000
  24/1000
  32/1000
  40/1000
  48/1000
  56/1000
  64/1000
  72/1000
  80/1000
  88/1000
  96/1000
  104/1000
  112/1000
  120/1000
  128/1000
  136/1000
  144/1000
  152/1000
  160/1000
  168/1000
  176/1000
  184/1000
  192/1000
  200/1000
  208/1000
  216/1000
  224/1000
  232/1000
  240/1000
  248/1000
  256/1000
  264/1000
  272/1000
  280/1000
  288/1000
  296/1000
  304/1000
  312/1000
  320/1000
  328/1000
  336/1000
  344/1000
  352/1000
  360/1000
  368/1000
  376/1000
  384/1000
  392/1000
  400/1000
  408/1000
  416/1000
  424/1000
  432/1000
  440/1000
  448/1000
  456/1000
  464/1000
  472/1000
  480/1000
  488/1000
  496/1000
  504/1000
  512/1000
  520/1000
  528/1000
  536/1000
  544/1000
  